已比较6版新提示词，采用第5版作为当前联合提炼默认候选。覆盖改善，仍有温度计误配、来源范围混用和引文失败，**尚未质量合格**。下方保存真实新旧图文及提示词实验对照。

### 按概念过滤图片：新实验结果

四模型、114张筛选前原图的独立对照见 [image_filter_models_debug.ipynb](/yzp/zhaozy/yangzepeng/0905/demiwtg/curation/v4/image_filter_models_debug.ipynb)。本批组合候选为Qwen3.8初筛→Gemma31B复核入选图，分歧暂缓；细物种身份问题仍未解决。**执行链已接入新组合；以下既有知识结果尚未重跑，仍是历史结果。**

### 当前正式执行链
原始datasets → 文档清洗/过滤/去重、图片字节检查 → 概念身份及文字相关性 → **Qwen3.8图片初筛 → Gemma31B独立复核入选图 → 分歧暂缓** → 原生图文关联与相似度补充 → 32K容量组装 → v5图文段落提炼 → 引用/像素核验 → 按需一次跨批整合 → 知识文件。

新RUN默认重新执行新图片筛选。`image_identity_definitions`可按concept_ref明确范围；未配置时使用原概念记录中的名称、别名、QID、简介等，剔除上游图片判定。补充limits为空不再误挡完整判断，但缺字段和协议矛盾仍暂缓。细物种身份仍需来源证据，双模型一致不等于已核验。

8001已有Gemma31时复用；默认`image_review_service='borrow'`自动借用并恢复本地GPU服务；`external`只使用外部已准备服务。这里只编排新链，历史展示仍标为旧结果。

原始材料、图片字节、模型和16K输出上限固定；清洗不重跑。每版3次联合提炼，v4/v5/v6继续核验且复用成功提炼。选择基于这3例人工对照，不代表全库最优。提示词源文件：`ops/prompts/joint_paragraphs.yaml`；实验版本：`ops/prompts/experiments/`。

# 概念原始材料 → 图文主题知识

原始 datasets 读入后按概念筛选，文档与图片分别清洗/筛选，再用原生图文关联、文本语义分组和图文相似度组织材料。模型共同读取原文和实际图片，输出主题内容；跨批只对候选做关系判断和必要的局部整合。

主线直接使用 demiflow Dataset；业务 op、prompt 在 `ops/`。最终展示 **标题 / 内容与图片 / 去重后的参考来源**，不展示审核状态。内部保留原文、引用、失败和审核记录。

逐算子调试入口：[玻璃棒 · 独立 cell](glass_operator_debug.ipynb)，从原始记录开始逐格运行并查看真实 Dataset。


### 主线算子：作用、输入、输出

|算子|输入|输出与作用|
|---|---|---|
|read_records → SelectSourceRecords → Concept/Document/ImageFromRecord|原始采集清单|按同一概念选择规则先过滤再转换，保留来源行号与源字段；QID页面映射仍完整保留，避免掩盖歧义|
|SelectConcept 与 join|概念过滤条件、三张表|选中概念及关联材料；不限制原始扫描条数|
|ReadDocument → CleanDocument → FilterDocumentBlocks|文档行、保存的原始页面|清洗正文块、原文位置、分离的图注和参考资料|
|CheckImage|图片行和原始文件|文件可用性、字节校验、尺寸等；缺图单独记录|
|PrepareIdentity → identity → ApplyIdentity|概念、材料预览|接受/排除材料及身份歧义；预览范围明确保留|
|BuildSourceBlocks → relevance → ApplyBlockSelection|身份接受的正文块|按概念相关性保留原文块，不改写原文|
|SelectAvailableImages → Qwen初筛 → Gemma独立复核 → ApplyConfirmedImageSelection|实际图片、概念、可选预标注|相关可用图、排除依据及未确定范围|
|PrepareRoutingMaterials|相关正文、图片、页面块中原生图片引用|材料表和可恢复的原生图文关联|
|EmbedParagraphBatch / EncodeImageTextMaterials|正文、实际图片|Qwen 文本向量、SigLIP2 图文向量，仅服务材料组织|
|RouteByTokenBudget → BuildRoutedJointRequest|向量、原生关联、容量配置|容量内的联合请求；没有匹配正文的图仍有独立处理机会|
|joint_paragraphs → ApplyParagraphs|原文、实际像素|主题段落、模型选择的相关互补图片、逐字文档引用或图片区域引用（image_refs）；无最终图片张数上限|
|verify_paragraphs → ApplyParagraphReview|提炼结果、同一原文与像素|逐块及整个标题/正文/图片组合的核验；移除上游接受理由，只看原始证据|
|PlanCrossBatchReview → BatchRelationshipReviews → review_relationships|同批与跨批段落及向量/引文|重复、互补、条件差异、冲突等候选关系；无候选则旁路|
|BuildLocalMergeGroups → merge_paragraphs → verify_merged_paragraphs|相关候选、原文、真实图片|局部整合内容和重新核验；超容量任务显式暂缓|
|ApplyLocalIntegration|原段落、局部整合结果|替换实际消费的块；内容移走的旧主题标记待重查|
|PrepareTopicRepairs → repair_topics → ApplyTopicRepairs|删文后失效的标题/正文/图片、原始证据|按需修复并再次看图核验；失败内部保存，不发布空标题纯图主题|
|RetainReviewedTopics → FormatTopicArticle|有效主题与来源表|三部分主题文章，发布阶段不按分数筛图|
|FinalKnowledgeRecord → checkpoint|概念、文档、图片、主题文章、过程审计|每概念一行的 knowledge_base.jsonl|

提示词源码在 `ops/prompts/`：`identity.yaml`、`relevance.yaml`、`select_images.yaml`、`joint_paragraphs.yaml`、`verify_paragraphs.yaml`、`review_relationships.yaml`、`merge_paragraphs.yaml`、`verify_merged_paragraphs.yaml`、`repair_topics.yaml`。state 内只保存冻结副本。

`quality_pipeline.py`中的`verify_topics`、`repair_topics`是原生Dataset链的组合函数，业务处理仍由上述op/prompt执行。实际试验仍有残留重复及文章割裂，不能因接通了算子就称质量合格。

默认只做一次按需整合；无候选即零关系调用。`global_material_audit=True`可显式开启全库未关联审计，开启时禁用入口下推。清洗不等于事实正确，首次核验与改写后的核验保留。


## 1．处理与查看配置
MAX_RECORDS默认None：每个指定文件不限制扫描条数；填写整数才截断。IDS、材料分批、每次模型材料预算和调用预算是独立配置，保持显式。默认view_saved只查看已有最终文件，不启动全量扫描。


In [ ]:
from pathlib import Path
import sys, asyncio, itertools, random, json
ROOT = Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from curation.v4.ops.filter_document_blocks import FilterDocumentBlocks
from curation.v4.ops.select_source_records import SelectSourceRecords
from curation.v4.ops.cross_batch import BatchRelationshipReviews, ApplyRelationshipReviews
from curation.v4.ops.multimodal import (SelectAvailableImages, BatchImageSelection, ApplyImageSelection,
    merge_image_decisions, SelectRelatedMaterials, BatchJointMaterials, ApplyJointExtraction,
    ApplyJointVerification, merge_joint_batches, merge_joint_scope, PrepareJointMerge, ApplyJointMerge)
from demiflow.standalone import local_data
from curation.v4.ops.material_routing import PrepareRoutingMaterials, RawPassageRows, EncodeImageTextMaterials, BuildRoutedJointRequest
from curation.v4.ops.paragraph_similarity import EmbedParagraphBatch, ParagraphRows
from curation.v4.ops.paragraphs import ApplyParagraphs, ApplyParagraphReview, SelectRetainedParagraphs
from curation.v4.ops.cross_batch import PlanCrossBatchReview, ApplyCrossBatchReview
from curation.v4.ops.paragraph_merge import ApplyParagraphMerge
from curation.v4.ops.topic_articles import TopicRows
from curation.v4.ops.token_routing import RouteByTokenBudget
from curation.v4.ops.paragraph_pipeline import (RouteConceptMaterials, PrepareVerifiedParagraphs, BuildLocalMergeGroups,
    ApplyLocalIntegration, SourceCatalog, FormatTopicArticle, FinalKnowledgeRecord)
# 业务算子：只处理数据，不隐藏上/下游Dataset。
from curation.v4.ops.dataset_operators import (
    ConceptFromRecord, DocumentFromRecord, ImageFromRecord, SelectConcept, MaterialLinks, ReadDocument, CleanDocument, CheckImage,
    CountMaterial, NestMaterial, merge_concept, merge_document, distinct,
    fill_material_counts, model_input)
from curation.v4.ops.prompt_operators import (
    SelectPassagesAndImages, BuildCandidateRecords,
    PrepareIdentity, ApplyIdentity, PrepareExtraction, ApplyExtraction,
    PrepareConsolidation, ApplyConsolidation, PrepareImageSupport, ApplyImageSupport)
from curation.v4.ops.fidelity import PrepareFidelity, ApplyFidelity
from curation.v4.ops.source_blocks import (BuildSourceBlocks, BatchSourceBlocks, ApplyBlockSelection,
    merge_block_decisions, BuildVerbatimCandidates, BatchSourceComparisons,
    ApplySourceComparison, merge_source_comparisons, ApplyComparedCandidates)
from curation.v4.ops.prompt_config import knowledge_prompt_pack, prompt_execution_options, save_prompt_config
# 下面仅为原文件发现、文件读取、版本冻结和提示词配置，不是流程对象。
from curation.v4.contracts import digest, run_lock, snapshot
from curation.v4.notebook_io import freeze_run, read_saved
from curation.v4.pipeline import DEFAULT
from curation.v4.ops.image_filter import (IMAGE_FILTER_DEFAULTS, RecordPrimaryImageSelection, PrepareImageReview, ApplyConfirmedImageSelection)
from curation.v4.image_filter_runtime import image_prompt_data, review_needed, save_image_filter_policy, validate_material_reuse
from curation.v4.local_review_service import image_review_service
from curation.v4.final_results import BuildKnowledgeRecord, view_knowledge

# view_saved只读已有结果；execute才执行下面唯一的pipeline入口。
# None 从原始数据开始；指定父run则复用已清洗、筛选材料及向量。
REUSE_MATERIALS = None  # 旧单模型入选材料不能跳过新筛选
MODE = 'view_saved'
RUN = ROOT / 'state/curation/v4/knowledge_dual_image_v1'
DATASET = ROOT / 'datasets/demiwtg'
# 以下参数仅用于新建run。全文件处理：IDS=None、SAMPLE_RATE=1、MAX_RECORDS=None。
# 取消这些工程预算不等于跨批整合、吞吐及质量已经验收。
IDS = ['legacy:玻璃棒', 'legacy:OK手势', 'legacy:白花芍药']  # 定向工程验证，不是随机抽样
SOURCE_SCOPE = 'collected'  # 三个原始采集文件；all再包含QID与Wiki文件
SAMPLE_RATE, SEED, MAX_RECORDS, GROUP_SIZE = 1.0, 42, None, 256
THROUGH = 'export'  # 可改identity/organize/extract/consolidate/fidelity/evidence/export
# 本轮三个概念的明确身份范围；扩量时从概念资料确定，勿按名称猜物种。
IMAGE_IDENTITY_DEFINITIONS = {'legacy:OK手势': '拇指和食指相触成环、其余手指伸展或放松的手势；相关图解和实际使用场景也可保留。', 'legacy:玻璃棒': '实验室中用于搅拌、引流等的实心玻璃棒；同属实验器材不自动属于目标。', 'legacy:白花芍药': '植物学物种 Paeonia sterniana；泛指白色芍药花或其他栽培品种不自动认证为这一物种。身份不确定时保留不确定性。'}
MODEL_CONFIG = {**IMAGE_FILTER_DEFAULTS, 'image_identity_definitions': IMAGE_IDENTITY_DEFINITIONS, 'joint_input_tokens':32768, 'text_mode':'multimodal', 'body_only':True, 'max_calls':None, 'max_output_tokens':16384,
                'temperature':0, 'timeout_s':900, 'block_unit_chars':1800,
                'block_batch_chars':8000, 'comparison_group_chars':16000,
                'text_embedding_model':str(ROOT.parent/'models/Qwen3-Embedding-0.6B'),
                'image_embedding_model':str(ROOT.parent/'models/siglip2-base-patch16-224')}  # 运行知识阶段时冻结本地模型配置与预算
# 修改代码/配置后执行须使用新的RUN目录，旧结果不可覆盖。

# 查看配置只影响展示，不影响处理或模型调用；新run实际sink为RUN/'knowledge_base.jsonl'。
FINAL_FILE = ROOT / 'state/curation/v4/prompt_coverage_v5_verified/knowledge_base.jsonl'
VIEW = dict(limit=10, concept_ids=None, sample_rate=1., seed=42,
            max_images=4, preview_width=200)  # 仅限制图片预览，全部图片记录仍在表内

from curation.v4.quality_pipeline import repair_topics


## 2．精简后的完整 pipeline

1. 读取原清单，按概念先过滤，再转换和关联文档/图片。
2. 清洗、去重、修复正文；检查图片文件。
3. 检查身份，筛选相关正文和图片。
4. 保留原生图文关联，按语义与容量组装材料。
5. 联合提炼，核对引用和实际图片。
6. 有候选才批量判断关系，最多一轮局部整合；只有改写部分重新核验。失效主题仍按需修复。
7. 保存图文知识及原始材料/过程审计。

默认不做全库未关联材料审计，不再固定第二轮整合，不再运行只记录残留关系的末轮模型检查。首次核验保留：清洗不保证模型改写或图片身份正确。

可在 MODEL_CONFIG 设置 `global_material_audit=True` 开启完整审计（会关闭入口下推）；`relationship_batch_pairs=8`、`relationship_batch_chars=24000` 控制关系请求容量。超容量不截断、不自动通过。

入口实测：同样26篇文档/322条图片，清洗全文逐篇一致，约100秒到达模型输入准备。上轮62对候选按新容量规则需要8次调用；新版本最终知识质量尚未端到端验证。下面仍为直接可见的demiflow Dataset编排。

批量协议实测：1次Qwen调用完整返回8对判断，ID与覆盖检查通过；仍全部判需整合，未证明语义质量改善。新全链尚未重跑。


In [ ]:
def run_pipeline(run, dataset, *, ids=None, sample_rate=1., seed=42,
                 max_records_per_source=None, group_size=32, through='gather',
                 model_config=None, project=ROOT, source_scope="all", reuse_preprocessing=None, reuse_materials=None):
    """Dataset编排直接在这里；业务算子只处理行，不决定上下游。"""
    if through not in ['gather','identity','organize','extract','consolidate','fidelity','evidence','export']:
        raise ValueError('Unknown stopping stage')
    run, dataset = Path(run), Path(dataset)
    if source_scope not in {"all", "collected"}: raise ValueError("invalid source_scope")
    config = {**DEFAULT, **IMAGE_FILTER_DEFAULTS, **(model_config or {})}
    # 全库未关联资料审计是可选旁路；开启时关闭入口下推以保证审计完整。
    global_audit = config.get('global_material_audit', False)
    if config.get('image_annotations_file'):
        config['image_annotations_sha256'] = digest(Path(config['image_annotations_file']).read_bytes())
    settings = dict(ids=ids, sample_rate=sample_rate, seed=seed,
                    max_records_per_source=max_records_per_source, group_size=group_size,
                    model_config=config, source_scope=source_scope)
    if reuse_preprocessing is not None:settings['reuse_preprocessing']=str(Path(reuse_preprocessing).resolve())
    if reuse_materials is not None:
        validate_material_reuse(reuse_materials, config)
        if through not in {'extract','consolidate','fidelity','evidence','export'} or config.get('text_mode')!='multimodal':
            raise ValueError('Material reuse begins at multimodal joint extraction')
        if reuse_preprocessing is not None:raise ValueError('Choose one reuse boundary')
        from curation.v4.notebook_io import frozen_material_inputs, import_frozen_materials
        settings['reuse_materials']=frozen_material_inputs(reuse_materials)
    # 只管并发锁与版本冻结，不隐藏任何业务调度；through可向后续跑。
    with run_lock(run):
        tables = run/'datasets'
        data = local_data()

        # 1. 文件快照仅供版本冻结和定位校验，不负责读取或调度。
        legacy_concepts_source = {'kind':'legacy_concepts', **snapshot(dataset/'meta/concepts.json')}
        qid_concepts_source = {'kind':'qid_concepts', **snapshot(dataset/'meta/qid_concepts.fat.jsonl.gz')}
        collected_documents_source = {'kind':'legacy_docs', **snapshot(dataset/'meta/docs.jsonl')}
        collected_images_source = {'kind':'legacy_images', **snapshot(dataset/'meta/images.jsonl')}
        wiki_pages_source = {'kind':'wiki_pages', **snapshot(dataset/'corpus/pages-en-part1.jsonl.gz')}
        active_sources=[legacy_concepts_source,collected_documents_source,collected_images_source]
        if source_scope=='all':active_sources += [qid_concepts_source,wiki_pages_source]
        version = freeze_run(run, dataset, active_sources, settings, run_pipeline, project)
        if reuse_preprocessing is not None:
            from curation.v4.notebook_io import reuse_preprocessing as import_preprocessing
            import_preprocessing(reuse_preprocessing,run,version,settings)

        if reuse_materials is not None:
            import_frozen_materials(settings['reuse_materials'],run,version)
        else:
            # 原生读取 → 保存原始解码行（含错误）→ 有效对象 → 业务字段转换。
            # read_records负责gzip/JSON解析、行号与扫描报告；map不读文件。
            # 解码失败或非对象行保留在read_*检查点，不静默丢弃原文。
            legacy_concepts_records = (data.read_records(dataset/'meta/concepts.json', format='json', item_prefix='concepts.item',
                max_records=max_records_per_source, missing='empty', report_path=run/'source_status/legacy_concepts.json')
                .filter(SelectSourceRecords('legacy_concepts', ids, sample_rate, seed, enabled=not global_audit))
                .checkpoint(tables/'read_legacy_concepts.jsonl', version=version))
            legacy_concepts = (legacy_concepts_records
                .filter(lambda r: r['error'] is None and isinstance(r['value'], dict))
                .map(ConceptFromRecord(legacy_concepts_source))
                .checkpoint(tables/'input_legacy_concepts.jsonl', version=version))

            if source_scope=='all':
                qid_concepts_records = (data.read_records(dataset/'meta/qid_concepts.fat.jsonl.gz',
                    max_records=max_records_per_source, missing='empty', report_path=run/'source_status/qid_concepts.json')
                    .filter(SelectSourceRecords('qid_concepts', ids, sample_rate, seed, enabled=not global_audit))
                .checkpoint(tables/'read_qid_concepts.jsonl', version=version))
                qid_concepts = (qid_concepts_records
                    .filter(lambda r: r['error'] is None and isinstance(r['value'], dict))
                    .map(ConceptFromRecord(qid_concepts_source))
                    .checkpoint(tables/'input_qid_concepts.jsonl', version=version))

            else:
                qid_concepts = data.from_iter(lambda: iter(()))

            collected_documents_records = (data.read_records(dataset/'meta/docs.jsonl',
                max_records=max_records_per_source, missing='empty', report_path=run/'source_status/collected_documents.json')
                .filter(SelectSourceRecords('legacy_docs', ids, sample_rate, seed, enabled=not global_audit))
                .checkpoint(tables/'read_collected_documents.jsonl', version=version))
            collected_documents = (collected_documents_records
                .filter(lambda r: r['error'] is None and isinstance(r['value'], dict))
                .map(DocumentFromRecord(collected_documents_source))
                .checkpoint(tables/'input_collected_documents.jsonl', version=version))

            collected_images_records = (data.read_records(dataset/'meta/images.jsonl',
                max_records=max_records_per_source, missing='empty', report_path=run/'source_status/collected_images.json')
                .filter(SelectSourceRecords('legacy_images', ids, sample_rate, seed, enabled=not global_audit))
                .checkpoint(tables/'read_collected_images.jsonl', version=version))
            collected_images = (collected_images_records
                .filter(lambda r: r['error'] is None and isinstance(r['value'], dict))
                .map(ImageFromRecord(collected_images_source))
                .checkpoint(tables/'input_collected_images.jsonl', version=version))

            if source_scope=='all':
                wiki_pages_records = (data.read_records(dataset/'corpus/pages-en-part1.jsonl.gz',
                    max_records=max_records_per_source, missing='empty', report_path=run/'source_status/wiki_pages.json')
                    .checkpoint(tables/'read_wiki_pages.jsonl', version=version))
                wiki_pages = (wiki_pages_records
                    .filter(lambda r: r['error'] is None and isinstance(r['value'], dict))
                    .map(DocumentFromRecord(wiki_pages_source))
                    .checkpoint(tables/'input_wiki_pages.jsonl', version=version))

            else:
                wiki_pages = data.from_iter(lambda: iter(()))

            concepts = (legacy_concepts.union(qid_concepts)
                .reduce_by_key('concept_ref', merge_concept)
                .checkpoint(tables/'concepts.jsonl', version=version))
            images = collected_images.checkpoint(tables/'images.jsonl', version=version)

            # 2. 页面关联：概念的lang/page_id → Wiki文档concept_refs。
            # left join保留未匹配页面；多概念对应不自动消歧。其他文档保留采集关联。
            page_refs = (concepts.flat_map(lambda c:c['page_refs'])
                .reduce_by_key(['lang','page_id','mapped_concept_ref'], distinct)
                .checkpoint(tables/'page_refs.jsonl', version=version))
            wiki_documents = (wiki_pages
                .join(page_refs, on=['lang','page_id'], how='left')
                .reduce_by_key('doc_id', merge_document))
            documents = collected_documents.union(wiki_documents)
            documents = documents.checkpoint(tables/'documents.jsonl', version=version)

            # 3. SelectConcept：概念行 → 增加selected/selection_reason，再filter。
            # 与读取端使用同一采样规则；下游业务算子不接收入口概念ID。
            concept_selection = (concepts.map(SelectConcept(ids, sample_rate, seed))
                .checkpoint(tables/'concepts_selected.jsonl', version=version))
            selected_concepts = concept_selection.filter(lambda c:c['selected'])
            selected_keys = selected_concepts.select_columns(['concept_ref'])
            if ids is not None:
                (data.from_iter(lambda:({'concept_ref':ref} for ref in ids))
                    .join(concepts.select_columns(['concept_ref']), on='concept_ref', how='anti')
                    .checkpoint(tables/'missing_concepts.jsonl', version=version))

            # 4. MaterialLinks：文档/图片行 → concept_ref与doc_id/image_id关联键。
            # 分别semi join入选概念，再用资料ID筛选原表；共享资料只处理一次。
            all_document_links = documents.flat_map(MaterialLinks('doc_id'))
            all_image_links = images.flat_map(MaterialLinks('image_id'))
            document_links = (all_document_links.join(selected_keys, on='concept_ref', how='semi')
                .checkpoint(tables/'documents_links.jsonl', version=version))
            image_links = (all_image_links.join(selected_keys, on='concept_ref', how='semi')
                .checkpoint(tables/'images_links.jsonl', version=version))
            selected_documents = (documents.join(document_links.select_columns(['doc_id'])
                .reduce_by_key('doc_id', distinct), on='doc_id', how='semi')
                .checkpoint(tables/'documents_selected.jsonl', version=version))
            selected_images = (images.join(image_links.select_columns(['image_id'])
                .reduce_by_key('image_id', distinct), on='image_id', how='semi')
                .checkpoint(tables/'images_selected.jsonl', version=version))
            # 未关联任何已读概念的资料另存；不是把未入选概念的资料判为无关。
            if global_audit:
                for objects, links, key, name in [
                    (documents, all_document_links, 'doc_id', 'documents'),
                    (images, all_image_links, 'image_id', 'images')]:
                    associated = (links.join(concepts.select_columns(['concept_ref']), on='concept_ref', how='semi')
                        .select_columns([key]).reduce_by_key(key, distinct))
                    objects.join(associated, on=key, how='anti').checkpoint(tables/f'{name}_unmatched.jsonl', version=version)


            # 5. ReadDocument → CleanDocument → FilterDocumentBlocks：读原文、解析正文、过滤与修复。
            # 原文及排除依据保留；重建clean_text和块定位后才交给身份/相关性判断。
            # 输入：选中文档path/sections；输出：raw_text → clean_text/块定位/准入状态。
            # 不删除原文，pending带原因保留；map_cached复用同输入同版本结果。
            processed_documents = (selected_documents
                .map_cached(ReadDocument(dataset), cache_dir=run/'cache/read_documents', version=version)
                .map_cached(CleanDocument(), cache_dir=run/'cache/clean_documents', version=version)
                .map_cached(FilterDocumentBlocks(), cache_dir=run/'cache/filter_documents', version=version)
                .checkpoint(tables/'documents_processed.jsonl', version=version))
            # 6. CheckImage：图片独立扩列；路径/哈希 → byte_status/byte_details。
            # 文件可用性检查不是图片语义核验。
            processed_images = (selected_images
                .map_cached(CheckImage(dataset), cache_dir=run/'cache/check_images', version=version)
                .checkpoint(tables/'images_processed.jsonl', version=version))

            # 可选：读取预标注JSONL冻结快照，按图片SHA关联；不把机器标签当人工核验。
            if config.get('image_annotations_file'):
                annotations = (data.read_records(config['image_annotations_file'])
                    .map(lambda r: {'sha256':r['value']['sha256'],'preannotation':r['value']} if r['error'] is None else (_ for _ in ()).throw(ValueError('invalid annotation row')))
                    .checkpoint(tables/'image_annotations.jsonl',version=version))
                processed_images = (processed_images.join(annotations,on='sha256',how='left')
                    .checkpoint(tables/'images_annotated.jsonl',version=version))

            # 7. CountMaterial：关联键join处理状态后，按概念归约计数。
            # 输出：每概念文档/图片总数、可读/字节通过数；零资料概念left join保留。
            document_counts = (document_links
                .join(processed_documents.select_columns(['doc_id','read_status']), on='doc_id')
                .reduce_by_key('concept_ref', CountMaterial('document_count','read_status','readable_documents')))
            image_counts = (image_links
                .join(processed_images.select_columns(['image_id','byte_status']), on='image_id')
                .reduce_by_key('concept_ref', CountMaterial('image_count','byte_status','verified_images')))
            concepts_ready = (selected_concepts
                .join(document_counts, on='concept_ref', how='left')
                .join(image_counts, on='concept_ref', how='left')
                .map(fill_material_counts)
                .checkpoint(tables/'concepts_ready.jsonl', version=version))

            # 8. NestMaterial + group_batches：到这里才按概念汇集完整文档/图片。
            # 输出knowledge_inputs：每行一个概念材料批次，含materials及批次索引。
            # 这只是执行分批，尚未完成跨批语义整合。
            concept_documents = document_links.join(
                processed_documents.map(NestMaterial('doc_id','documents')), on='doc_id')
            concept_images = image_links.join(
                processed_images.map(NestMaterial('image_id','images')), on='image_id')
            material_batches = concept_documents.union(concept_images).group_batches(
                'concept_ref', max_rows=group_size, output='materials')
            batches = (concepts_ready.join(material_batches, on='concept_ref', how='left')
                .checkpoint(tables/'knowledge_inputs.jsonl', version=version))
            if through == 'gather': return batches

        # 9. ResolveIdentity：概念与清洗资料 → 身份/逐材料依据/接受拒绝/未查看范围。
        # model_input仅转换联合输入格式；身份歧义blocked，图片此时仅看元数据。
        knowledge_run = run/'knowledge'
        pack, prompt_text = knowledge_prompt_pack(config)
        options = prompt_execution_options(run, config)
        save_prompt_config(run, prompt_text, options)
        prompt_data = local_data(prompt_packs={'knowledge.yaml':pack},
                                 max_prompt_requests=config['max_calls'], prompt_options=options)
        # 同一材料Dataset进入原生提示词执行上下文；请求预算和持久账本跨阶段共享。
        if reuse_materials is None:
            batches = prompt_data.read_json(str(tables/'knowledge_inputs.jsonl'))
            identified = (batches.map(model_input)
                # 准备：保留源资料，生成identity_prompt；不调用模型。
                .map_cached(PrepareIdentity(knowledge_run, config),
                            cache_dir=knowledge_run/'cache/PrepareIdentity', version=version)
                # 调用：demiflow负责异步HTTP、完整请求响应、预算和精确回放。
                .map_prompt_async('identity', config='knowledge.yaml', inputs={'payload':'identity_prompt'},
                                  output='prompt_result', call_output='prompt_call', error_output='prompt_error',
                                  when=lambda r: not r.get('blocked') and 'identity_prompt' in r,
                                  concurrency=1, queue_depth=1)
                # 校验：结果与原文/材料对应检查；错误保留为blocked，不丢行。
                .map_cached(ApplyIdentity(knowledge_run, config),
                            cache_dir=knowledge_run/'cache/ApplyIdentity', version=version)
                .checkpoint(tables/'knowledge_identity.jsonl', version=version))
            if through == 'identity': return identified

        # 10M. 相关材料筛选 → 图文联合提炼 → 引用/像素核验 → 跨批候选去重与冲突处理。
        # 以下分批只控制单次输入；文字组×图片组完整遍历，不按已有知识找图片。
        if config.get('text_mode') == 'multimodal':
            if config.get('integration_rounds',1) != 1: raise ValueError('Slim pipeline performs one conditional integration pass; further repair needs a separate reviewed run')
            if reuse_materials is not None:
                related=prompt_data.read_json(str(tables/'related_materials.jsonl'))
                routing_materials=prompt_data.read_json(str(tables/'routing_materials.jsonl'))
                text_embeddings=(prompt_data.read_json(str(tables/'material_text_embeddings.jsonl'))
                    .flat_map(lambda r:r['items']).reduce_by_key('case_id',
                    lambda acc,r:{'case_id':r['case_id'],'passage_embeddings':{**acc['passage_embeddings'],r['source_id']:r}},initial={'passage_embeddings':{}}))
                image_text_embeddings=prompt_data.read_json(str(tables/'material_image_embeddings.jsonl'))
            else:
                blocks = (identified.map(BuildSourceBlocks(config.get('block_unit_chars',1800), body_only=True))
                    .map(SelectAvailableImages()).checkpoint(tables/'multimodal_materials.jsonl', version=version))
                text_requests = blocks.flat_map(BatchSourceBlocks(config.get('block_batch_chars',8000)))
                text_decisions = (text_requests.map_prompt_async('select_blocks', config='knowledge.yaml',
                    inputs={'payload':'block_prompt'}, output='prompt_result',call_output='prompt_call',error_output='prompt_error',concurrency=1,queue_depth=1)
                    .map_cached(ApplyBlockSelection(relevance_only=True),cache_dir=knowledge_run/'cache/text_relevance',version=version)
                    .checkpoint(tables/'text_relevance.jsonl',version=version)
                    .reduce_by_key('case_id',merge_block_decisions))
                # 10A. Qwen初筛：只给概念身份资料和像素，不给上游图片结论。
                image_requests = blocks.flat_map(BatchImageSelection(config.get('image_batch_size',4),
                    config['image_identity_definitions'], neutral=True)).checkpoint(tables/'image_requests.jsonl',version=version)
                primary_data = image_prompt_data(run, config)
                primary = (primary_data.read_json(str(tables/'image_requests.jsonl'))
                    .map_prompt_async('select_images',config='knowledge.yaml',
                        inputs={'payload':'image_prompt','images':'pixel_images'},output='prompt_result',call_output='prompt_call',error_output='prompt_error',concurrency=1,queue_depth=1)
                    .map_cached(RecordPrimaryImageSelection(),cache_dir=run/'cache/image_primary',version=version)
                    .checkpoint(tables/'image_primary.jsonl',version=version))
                # 10B. 含入选图才复核；保留整个原批次，Gemma看不到Qwen判断。
                review_rows = primary.map(PrepareImageReview()).checkpoint(tables/'image_review_inputs.jsonl',version=version)
                review_data = image_prompt_data(run, config, review=True)
                review_requests = review_data.read_json(str(tables/'image_review_inputs.jsonl')).filter(lambda r:r['review_required'])
                review_path = tables/'image_review_responses.jsonl'
                # 只在有未完成复核时借用GPU；该阶段落盘后恢复Qwen及预标注。
                with image_review_service(run, config, needed=review_needed(review_requests,review_path,version)):
                    reviewed = (review_requests.map_prompt_async('select_images',config='knowledge.yaml',
                        inputs={'payload':'image_prompt','images':'pixel_images'},output='prompt_result',call_output='prompt_call',error_output='prompt_error',
                        concurrency=config['image_review_concurrency'],queue_depth=config['image_review_concurrency'])
                        .checkpoint(review_path,version=version))
                # 10C. 原排除/待定继续保留；入选图须独立复核keep，否则暂缓。
                image_decisions = (reviewed.union(review_rows.filter(lambda r:not r['review_required']))
                    .map_cached(ApplyConfirmedImageSelection(),cache_dir=run/'cache/image_confirmed',version=version)
                    .checkpoint(tables/'image_relevance.jsonl',version=version)
                    .reduce_by_key('case_id',merge_image_decisions))
                related = (blocks.join(text_decisions,on='case_id',how='left')
                    .join(image_decisions,on='case_id',how='left').map(SelectRelatedMaterials())
                    .checkpoint(tables/'related_materials.jsonl',version=version))
                save_image_filter_policy(run,config)
                if through=='organize':return related
                # 11. 原生图文关联 → 文本embedding分组 → 图文embedding补充关联。
                routing_materials = related.map(PrepareRoutingMaterials()).checkpoint(tables/'routing_materials.jsonl',version=version)
                text_embeddings = (routing_materials.flat_map(RawPassageRows())
                    .group_batches('embedding_bucket',max_rows=2,output='items')
                    .map_cached(EmbedParagraphBatch(config['text_embedding_model']),cache_dir=run/'cache/material_text_embedding',version=version)
                    .checkpoint(tables/'material_text_embeddings.jsonl',version=version)
                    .flat_map(lambda r:r['items']).reduce_by_key('case_id',
                        lambda acc,r:{'case_id':r['case_id'],'passage_embeddings':{**acc['passage_embeddings'],r['source_id']:r}},initial={'passage_embeddings':{}}))
                image_text_embeddings = (routing_materials
                    .map_cached(EncodeImageTextMaterials(config['image_embedding_model']),cache_dir=run/'cache/material_image_embedding',version=version)
                    .checkpoint(tables/'material_image_embeddings.jsonl',version=version))
            if reuse_materials is not None:save_image_filter_policy(run,config)
            routed = (routing_materials.join(text_embeddings,on='case_id',how='left')
                .join(image_text_embeddings.select_columns(['case_id','text_windows','image_vectors']),on='case_id',how='left')
                .map(RouteByTokenBudget(ROOT.parent/'models/Qwen3.8-27B', config.get('joint_input_tokens',32768)))
                .checkpoint(tables/'material_routing.jsonl',version=version))
            # 12. 容量组装，不做文字×图片全组合；图片数量是单次输入目标，非最终保留上限。
            joint_requests = (routed.flat_map(lambda r:r['requests']).map(BuildRoutedJointRequest())
                .checkpoint(tables/'joint_requests.jsonl',version=version))
            # 13. 真实像素联合提炼：模型负责选图、重复取舍、条件和引文。
            extracted = (joint_requests.map_prompt_async('joint_paragraphs',config='knowledge.yaml',
                inputs={'payload':'joint_prompt','images':'pixel_images'},output='prompt_result',call_output='prompt_call',error_output='prompt_error',concurrency=1,queue_depth=1)
                .map_cached(ApplyParagraphs(),cache_dir=run/'cache/paragraph_extract',version=version)
                .checkpoint(tables/'paragraph_extract.jsonl',version=version))
            if through=='extract':return extracted
            verified = (extracted.map_prompt_async('verify_paragraphs',config='knowledge.yaml',
                inputs={'payload':'verify_payload','images':'pixel_images'},output='prompt_result',call_output='prompt_call',error_output='prompt_error',concurrency=1,queue_depth=1)
                .map_cached(ApplyParagraphReview(),cache_dir=run/'cache/paragraph_verify',version=version)
                .checkpoint(tables/'paragraph_verify.jsonl',version=version))
            # 14. 把已核验段落与实际请求对齐，独立内容继续保留。
            originals = (verified.join(joint_requests.select_columns(['batch_id','pixel_images']),on='batch_id',how='left')
                .map_cached(PrepareVerifiedParagraphs(run), cache_dir=run / "cache/local_verified_rows", version=version).checkpoint(tables/'paragraph_originals.jsonl',version=version))
            # 14A. 仅修复删文后失效/缺正文/核验未通过的主题；内部仍是原生demiflow链。
            originals, initial_repairs = repair_topics(originals,run=run,version=version,stage='initial')
            all_joint_requests = joint_requests.union(initial_repairs)
            # 15—16. 同批/跨批候选批量判断，只做一次按需整合；仅改写内容再核验。
            round_tables = tables / 'integration_1'
            original_groups = originals.reduce_by_key('concept',lambda acc,r:{'concept':r['concept'],'source_rows':acc['source_rows']+[r]},initial={'source_rows':[]})
            public_rows = originals.map(SelectRetainedParagraphs()).map(lambda r:r['content'])
            paragraph_embeddings = (public_rows.flat_map(ParagraphRows()).group_batches('embedding_bucket',max_rows=2,output='items')
                .map_cached(EmbedParagraphBatch(config['text_embedding_model']),cache_dir=run/'cache/output_text_embedding',version=version)
                .checkpoint(round_tables/'paragraph_embeddings.jsonl',version=version).flat_map(lambda r:r['items']))
            paragraph_groups = paragraph_embeddings.reduce_by_key('concept',lambda acc,r:{'concept':r['concept'],'items':acc['items']+[r]},initial={'items':[]})
            plans = paragraph_groups.map(PlanCrossBatchReview(include_same_batch=True, skip_single_batch=True)).checkpoint(round_tables/'cross_batch_plan.jsonl',version=version)
            # 15. 仅对同批/跨批的候选对判断关系，再决定是否局部整合。
            cross_reviews = (plans.flat_map(BatchRelationshipReviews(config.get('relationship_batch_pairs',8), config.get('relationship_batch_chars',24000)))
                .map_prompt_async('review_relationships',config='knowledge.yaml',inputs={'payload':'review_payload'},
                    when=lambda r:not r.get('batch_error'), output='prompt_result',call_output='prompt_call',error_output='prompt_error',concurrency=1,queue_depth=1)
                .map_cached(ApplyRelationshipReviews(),cache_dir=run/'cache/cross_review',version=version)
                .checkpoint(round_tables/'cross_batch_review_batches.jsonl',version=version)
                .flat_map(lambda r:r['pair_reviews'])
                .checkpoint(round_tables/'cross_batch_reviews.jsonl',version=version)
                .reduce_by_key('concept',lambda acc,r:{'concept':r['concept'],'review_rows':acc['review_rows']+[r]},initial={'review_rows':[]}))
            # 16. 协调交叠候选，再建互不覆盖的局部整合任务；原文/像素随任务携带。
            local_groups = (paragraph_groups.join(cross_reviews,on='concept',how='left')
                .join(original_groups,on='concept',how='left').map(BuildLocalMergeGroups())
                .checkpoint(round_tables/'local_merge_plan.jsonl',version=version))
            local_requests = local_groups.flat_map(lambda r:r['requests']).checkpoint(round_tables/'local_merge_requests.jsonl',version=version)
            local_extracted = (local_requests.map_prompt_async('merge_paragraphs',config='knowledge.yaml',
                inputs={'payload':'merge_payload','images':'pixel_images'},output='prompt_result',call_output='prompt_call',error_output='prompt_error',concurrency=1,queue_depth=1)
                .map_cached(ApplyParagraphMerge(),cache_dir=run/'cache/local_merge',version=version)
                .checkpoint(round_tables/'local_merge_extract.jsonl',version=version))
            # 改写后的文字、标题与图片关联再次对照原文和真实像素核验。
            local_verified = (local_extracted.map_prompt_async('verify_merged_paragraphs',config='knowledge.yaml',
                inputs={'payload':'verify_payload','images':'pixel_images'},output='prompt_result',call_output='prompt_call',error_output='prompt_error',concurrency=1,queue_depth=1)
                .map_cached(ApplyParagraphReview(),cache_dir=run/'cache/local_merge_verify',version=version)
                .map_cached(PrepareVerifiedParagraphs(run), cache_dir=run / "cache/local_verified_rows", version=version)
                .checkpoint(round_tables/'local_merge_verify.jsonl',version=version)
                .join(local_requests.select_columns(['batch_id','pixel_images']),on='batch_id',how='left'))
            local_results = local_verified.reduce_by_key('concept',lambda acc,r:{'concept':r['concept'],'local_results':acc['local_results']+[r]},initial={'local_results':[]})
            assembled = (original_groups.join(local_results,on='concept',how='left').map(ApplyLocalIntegration())
                .checkpoint(round_tables/'paragraph_assembly.jsonl',version=version))
            originals, repaired = repair_topics(assembled.flat_map(lambda r:r['rows']),run=run,version=version,stage='integration_1')
            all_joint_requests = all_joint_requests.union(local_requests).union(repaired)
            # 不再调用模型只为记录残留关系；一次整合不保证完全去重，后续按实际质量问题定向复查。
            final_rows = originals.checkpoint(run/'paragraphs.jsonl',version=version)
            all_joint_requests.checkpoint(run/'requests.jsonl',version=version)
            if through in {'consolidate','fidelity','evidence'}:return final_rows
            # 17. 发布只组织三部分，不再用相似度或张数筛图。
            catalogs = related.map(SourceCatalog()).checkpoint(tables/'source_catalog.jsonl',version=version)
            articles = (final_rows.map(SelectRetainedParagraphs()).map(lambda r:r['content']).flat_map(TopicRows())
                .join(catalogs,on='concept',how='left').map(FormatTopicArticle())
                .checkpoint(tables/'articles.jsonl',version=version))
            concept_articles = articles.reduce_by_key('concept',lambda acc,r:{'concept':r['concept'],'articles':acc['articles']+[r['article']]},initial={'articles':[]})
            # 18. 知识sink保留概念、原始资料、图片、最终主题内容及过程审计。
            audit = local_groups.map(lambda r:{'concept':r['concept'],'local_audit':{'pending':r['pending'],'local_task_count':len(r['requests'])}})
            plan_audit = plans.map(lambda r:{'concept':r['concept'],'cross_batch_plan':{'pending':r['pending'],'candidate_count':len(r['review_requests']),'passthrough_paragraph_ids':r['passthrough_paragraph_ids']}})
            return (related.map(lambda r:{**r,'concept':r['identity']['target_label']})
                .join(concept_articles,on='concept',how='left').join(audit,on='concept',how='left').join(plan_audit,on='concept',how='left')
                .map(FinalKnowledgeRecord()).checkpoint(run/'knowledge_base.jsonl',version=version))

        # 10A. 原文优先试验：替代生成式提取/忠实性改写审核，仍使用同一Dataset。
        # 1800/8000/16000是单块/请求的软目标，超长完整块单独处理，不截断文档总量。
        # 本分支当前验收文字；原始图片保留，新的知识尚未做像素支持核验。
        if config.get('text_mode') == 'source_blocks':
            # 身份接受文档 → 全部保留块、章节、相邻上下文、原文定位和未处理范围。
            blocks = (identified.map(BuildSourceBlocks(config.get('block_unit_chars',1800), body_only=config.get('body_only',False)))
                .checkpoint(tables/'source_blocks.jsonl', version=version))
            requests = (blocks.flat_map(BatchSourceBlocks(config.get('block_batch_chars',8000)))
                .checkpoint(tables/'block_requests.jsonl', version=version))
            # 模型只选择/暂缓/排除ID，不负责重写原文或生成引文。
            selected_batches = (requests
                .map_prompt_async('select_blocks', config='knowledge.yaml', inputs={'payload':'block_prompt'},
                    output='prompt_result', call_output='prompt_call', error_output='prompt_error',
                    concurrency=1, queue_depth=1)
                .map_cached(ApplyBlockSelection(relevance_only=config.get('relevance_only', True)), cache_dir=knowledge_run/'cache/ApplyBlockSelection', version=version)
                .checkpoint(tables/'block_decisions.jsonl', version=version))
            decisions = selected_batches.reduce_by_key('case_id', merge_block_decisions)
            candidates = (blocks.join(decisions, on='case_id', how='left')
                # statement与quote直接从原文块复制，条件保留在原文内，空conditions不表示无条件。
                .map(BuildVerbatimCandidates())
                .checkpoint(tables/'verbatim_candidates.jsonl', version=version))
            if through in {'organize','extract'}: return candidates
            # 分组之间两两比较：覆盖本材料批的全部候选对，不靠主题标签漏掉差异。
            # 比较仍是模型判断；分组数增加会带来二次方请求量，尚非全量吞吐验收。
            comparisons = (candidates.flat_map(BatchSourceComparisons(config.get('comparison_group_chars',16000)))
                .checkpoint(tables/'comparison_requests.jsonl', version=version))
            compared = (comparisons
                .map_prompt_async('compare_blocks', config='knowledge.yaml', inputs={'payload':'compare_prompt'},
                    output='prompt_result', call_output='prompt_call', error_output='prompt_error',
                    concurrency=1, queue_depth=1)
                .map_cached(ApplySourceComparison(), cache_dir=knowledge_run/'cache/ApplySourceComparison', version=version)
                .checkpoint(tables/'source_comparisons.jsonl', version=version))
            comparison_results = compared.reduce_by_key('case_id', merge_source_comparisons)
            reviewed = (candidates.join(comparison_results, on='case_id', how='left')
                .map(ApplyComparedCandidates())
                .checkpoint(tables/'knowledge_block_review.jsonl', version=version))
            if through in {'consolidate','fidelity','evidence'}: return reviewed
            exported = (reviewed.map_cached(BuildCandidateRecords(knowledge_run,config),
                cache_dir=knowledge_run/'cache/BuildCandidateRecords', version=version)
                .checkpoint(tables/'knowledge_export.jsonl', version=version))
            return exported.map(BuildKnowledgeRecord()).checkpoint(run/'knowledge_base.jsonl', version=version)

        # 10. SelectPassagesAndImages：接受资料 → 去重、选完整章节块及可用图。
        # 输出material_pack：passages/images/duplicates/omissions/image_gaps。
        selected_materials = (identified
            .map_cached(SelectPassagesAndImages(knowledge_run, config),
                        cache_dir=knowledge_run/'cache/SelectPassagesAndImages', version=version)
            .checkpoint(tables/'knowledge_organize.jsonl', version=version))
        if through == 'organize': return selected_materials

        # 11. ExtractKnowledge：本批多份passages → extraction。
        # facts每项含statement/conditions/exceptions/evidence；缺证与争议暂缓。
        extracted = (selected_materials
            # 准备：保留源资料，生成extract_prompt；不调用模型。
            .map_cached(PrepareExtraction(knowledge_run, config),
                        cache_dir=knowledge_run/'cache/PrepareExtraction', version=version)
            # 调用：demiflow负责异步HTTP、完整请求响应、预算和精确回放。
            .map_prompt_async('extract', config='knowledge.yaml', inputs={'payload':'extract_prompt'},
                              output='prompt_result', call_output='prompt_call', error_output='prompt_error',
                              when=lambda r: not r.get('blocked') and 'extract_prompt' in r,
                              concurrency=1, queue_depth=1)
            # 校验：结果与原文/材料对应检查；错误保留为blocked，不丢行。
            .map_cached(ApplyExtraction(knowledge_run, config),
                        cache_dir=knowledge_run/'cache/ApplyExtraction', version=version)
            .checkpoint(tables/'knowledge_extract.jsonl', version=version))
        if through == 'extract': return extracted

        # 12. ConsolidateKnowledge：extraction + 原片段 → knowledge。
        # 比较重复/互补/条件/矛盾，保留changes，未解冲突不混入保留候选。
        reviewed = (extracted
            # 准备：保留源资料，生成consolidate_prompt；不调用模型。
            .map_cached(PrepareConsolidation(knowledge_run, config),
                        cache_dir=knowledge_run/'cache/PrepareConsolidation', version=version)
            # 调用：demiflow负责异步HTTP、完整请求响应、预算和精确回放。
            .map_prompt_async('consolidate', config='knowledge.yaml', inputs={'payload':'consolidate_prompt'},
                              output='prompt_result', call_output='prompt_call', error_output='prompt_error',
                              when=lambda r: not r.get('blocked') and 'consolidate_prompt' in r,
                              concurrency=1, queue_depth=1)
            # 校验：结果与原文/材料对应检查；错误保留为blocked，不丢行。
            .map_cached(ApplyConsolidation(knowledge_run, config),
                        cache_dir=knowledge_run/'cache/ApplyConsolidation', version=version)
            .checkpoint(tables/'knowledge_consolidate.jsonl', version=version))
        if through == 'consolidate': return reviewed

        # 13. 审核陈述是否忠实于原文：knowledge（含暂缓项）+ 完整入选片段 → fidelity_reviews。
        # 不改写陈述；不支持/不确定的保留项转暂缓；此前暂缓项不会因这一步通过而恢复。
        faithful = (reviewed
            .map_cached(PrepareFidelity(knowledge_run, config),
                        cache_dir=knowledge_run/'cache/PrepareFidelity', version=version)
            .map_prompt_async('fidelity', config='knowledge.yaml', inputs={'payload':'fidelity_prompt'},
                              output='prompt_result', call_output='prompt_call', error_output='prompt_error',
                              when=lambda r: not r.get('blocked') and 'fidelity_prompt' in r,
                              concurrency=1, queue_depth=1)
            .map_cached(ApplyFidelity(knowledge_run, config),
                        cache_dir=knowledge_run/'cache/ApplyFidelity', version=version)
            .checkpoint(tables/'knowledge_fidelity.jsonl', version=version))
        if through == 'fidelity': return faithful

        # 14. CheckImageSupport：knowledge.facts + 实际图片 → image_evidence。
        # 逐图×知识记录区域、支持范围和局限，缺图/none与未执行分别记录。
        supported = (faithful
            # 准备：保留源资料，生成evidence_prompt；不调用模型。
            .map_cached(PrepareImageSupport(knowledge_run, config),
                        cache_dir=knowledge_run/'cache/PrepareImageSupport', version=version)
            # 调用：demiflow负责异步HTTP、完整请求响应、预算和精确回放。
            .map_prompt_async('evidence', config='knowledge.yaml', inputs={'payload':'evidence_prompt', 'images':'evidence_images'},
                              output='prompt_result', call_output='prompt_call', error_output='prompt_error',
                              when=lambda r: not r.get('blocked') and 'evidence_prompt' in r,
                              concurrency=1, queue_depth=1)
            # 校验：结果与原文/材料对应检查；错误保留为blocked，不丢行。
            .map_cached(ApplyImageSupport(knowledge_run, config),
                        cache_dir=knowledge_run/'cache/ApplyImageSupport', version=version)
            .checkpoint(tables/'knowledge_evidence.jsonl', version=version))
        if through == 'evidence': return supported

        # 15. BuildCandidateRecords：累计结果 → export（机器候选、暂缓、补证任务）。
        # 不升级为人工核验知识；文档/图片/原始请求响应仍可追溯。
        result = (supported
            .map_cached(BuildCandidateRecords(knowledge_run, config),
                        cache_dir=knowledge_run/'cache/BuildCandidateRecords', version=version)
            .checkpoint(tables/'knowledge_export.jsonl', version=version))
        # 16. 最终文件sink：每行一个概念材料批，原始资料与知识分层保留。
        # checkpoint是demiflow原生文件终结操作：原子写JSONL、版本检查、断点复用。
        knowledge_base = (result.map(BuildKnowledgeRecord())
            .checkpoint(run/'knowledge_base.jsonl', version=version))
        return knowledge_base


# 唯一执行入口；run/dataset/sources是显式参数，没有f/StreamFlow。
if MODE == 'execute':
    final_dataset = await asyncio.to_thread(
        run_pipeline, RUN, DATASET, ids=IDS, sample_rate=SAMPLE_RATE,
        seed=SEED, max_records_per_source=MAX_RECORDS, group_size=GROUP_SIZE,
        through=THROUGH, model_config=MODEL_CONFIG, source_scope=SOURCE_SCOPE, reuse_materials=REUSE_MATERIALS)
else:
    final_dataset = None  # 只读模式由下方最终结果查看器读取FINAL_FILE
print(f'模式：{MODE}；主链终点：{THROUGH}；结果目录：{RUN}')
if MODE == 'execute' and THROUGH == 'export': FINAL_FILE = RUN/'knowledge_base.jsonl'


## 3．提示词优化后真实结果与原32K版对照

左侧原版，右侧第5版提炼并核验后的结果。正文覆盖恢复，但温度计误配等问题仍在；本表是待审核机器结果。核验run只记录核验/修复调用，提炼3次记录在父run prompt_coverage_v5。查看不调用模型。

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from curation.v4.pipeline_comparison import show_run_comparison
PREVIOUS_RUN = ROOT / 'state/curation/v4/token32k_reuse_three_v2'
RESULT_RUN = ROOT / 'state/curation/v4/prompt_coverage_v5_verified'
from IPython.display import display, Markdown
import pandas as pd
experiment = json.loads((ROOT/'state/curation/v4/prompt_optimization_v1/comparison.json').read_text())
display(Markdown(experiment['scope']))
display(pd.DataFrame(experiment['rows']).style.set_properties(**{'white-space':'pre-wrap','text-align':'left'}))
show_run_comparison(PREVIOUS_RUN, RESULT_RUN, concepts=None, images=True)

## 4．历史段落相似度实验（可选对照）

下页是先前 30 段输出的冻结实验，不是本轮结果。当前自动候选判断及局部整合已接入上方主链；本轮中间数据见 RUN/datasets/cross_batch_plan.jsonl、cross_batch_reviews.jsonl、local_merge_plan.jsonl。


In [ ]:
from IPython.display import HTML, display
SIMILARITY_PREVIEW = ROOT / "state/curation/v4/paragraph_similarity_review_v2/preview.html"
if globals().get("SHOW_DEBUG_EXPERIMENTS", False): display(HTML(SIMILARITY_PREVIEW.read_text()))


## 5．图文匹配过程（可选调试）
同一批56段正文、23张图片：计划联合提炼62组→18组。本轮已补齐18组；这里保留首轮分组对比和5组样例作历史对照。最终跨组整合内容请看第3节。
相似度只决定材料优先同批，不认证图片身份或知识支持。代码见ops/material_routing.py，编排见try_material_routing.py。


In [ ]:
from IPython.display import HTML, display
# routing看图片对应原文与分组；knowledge看5组实际提炼的保留内容。
ROUTING_VIEW = "routing"
ROUTING_PAGES = {
 "routing": "state/curation/v4/material_routing_review_v2/preview.html",
 "knowledge": "state/curation/v4/material_routed_results_v1/preview.html",
}
if globals().get("SHOW_DEBUG_EXPERIMENTS", False): display(HTML((ROOT / ROUTING_PAGES[ROUTING_VIEW]).read_text()))


## 字段参考（按需展开）

入口列随算子保留，下方入口字段仍适用；旧 facts/evidence 分支字段用于历史对照。当前主线最终行：concept 为概念名，case_id 为本材料批引用，identity 为身份判断，documents/images 为带原始记录和处理信息的资料，knowledge 为主题文章列表，audit 为材料筛选与跨批处理记录。knowledge 每项包含 title、content（paragraphs 与 images）、references；图片字段含 image_id/caption/region/limitations，来源字段含 title/url/source_ids/kinds。具体原文引句、暂缓块及模型判断保存在 paragraphs.jsonl 和 datasets 中，不混入公共正文。

<details>
<summary>1．读取原始文件，分别形成概念、文档、图片 Dataset</summary>

输入是 `datasets` 的采集文件，而非历史 clean_docs。每个文件按实际 schema 读取；同类文件才合并。Wiki 文档的 `lang + page_id` 与概念来源中的对应页面关联。源行读错、来源缺失、读取预算截断分别记录。

输出是三个 Dataset 的原始列；来源字段保留，新增处理字段保留明确含义：

| Dataset | 字段 | 含义 |
|---|---|---|
| concepts | concept_ref | `legacy:概念名` 或 `qid:QID`，当前来源概念引用，不是已消歧身份，也不迁移权威主键 |
| concepts | name / aliases / qid | 展示名、别名、外部 QID；完整原始身份字段另见 source_records |
| concepts | source_records | 原始身份行列表，每项 fields 是源字段，source 是定位 |
| concepts | identity_status | source_only 表示只沿用来源关联，未经身份核验 |
| concepts | page_refs | 原始 sitelink 展开的 lang、page_id、mapped_concept_ref；仅用于页面关联 |
| documents | doc_id | 来源版本及行内容生成的文档引用；不代表内容去重后的永久文档 ID |
| documents | concept_refs | 采集标签或页面对应得到的概念引用列表；允许共享资料 |
| documents | title / url / path | 原始标题、页面地址、已保存正文路径（原源不存在时可缺省） |
| documents | format / sections | saved_text 为已下载文本；wiki_sections 的正文在原始章节列表中 |
| documents | lang / page_id / source_qid | Wiki 语言、页面号（字符串）、来源自带 QID |
| documents | association_status | source_only 来源关联；unassociated 无关联；ambiguous_mapping 页面有多个待消歧对应 |
| images | image_id / concept_refs | 来源图片引用及采集时关联概念列表 |
| images | path / sha256 / url / caption | 原始字节位置、来源哈希、来源 URL、来源图注；均不能替代看图 |
| documents、images | source | 原始来源定位；其 path 为元数据文件，row 为从 1 开始的位置，snapshot 为文件版本，content_sha256 为源行对象哈希 |

文档、图片其余原始列原样保留，具体字段名随来源变化；概念原始列在 source_records.fields 完整保留。缺省字段不是空值事实。`page_refs`、关联键及异常文件属于计算辅助数据，无须作为业务主线逐表理解。

</details>

<details>
<summary>2．筛选／采样概念，再关联文档和图片</summary>

输入：三个 Dataset 的原始列。概念先通过 ID 过滤、固定种子采样，输出增加 `selected`（是否入选）、`selection_reason`（selected、id_filter、concept_sample）。不设置 ids、sample_rate=1、取消读取上限，即使用同一条全量逻辑。

文档／图片分别将 concept_refs 展开为临时关联键，再 `.join(selected_concepts, how="semi")`。共享文档仍只保留一行供清洗。不会把 docs × images 联成大表。

输出 documents_selected、images_selected 保留原对象字段；documents_links / images_links 只含 concept_ref 和 doc_id / image_id。未匹配任何已读概念或关联歧义的资料保存在 *_unmatched，不丢弃；请求但没读到的概念在 missing_concepts。

</details>

<details>
<summary>3．读取下载正文 → 清洗文本外壳：文档连续扩列</summary>

输入：documents_selected；其中 path 或 sections 提供原文。

| 算子 | 新增输出字段 | 含义 |
|---|---|---|
| ReadDocument | raw_text / raw_sha256 | 完整读到的文本及读取字节的哈希；Wiki 是章节串接文本的哈希 |
| ReadDocument | read_status / read_error | readable 或 read_error；失败原因单独保存 |
| CleanDocument | clean_text / clean_version | 规则清洗文本及规则版本，不等于内容可靠 |
| CleanDocument | clean_status / clean_warnings | 清洗候选、待检查、不可用等状态及原因 |
| CleanDocument | clean_blocks | 块级原文与清洗文本的定位：block_id、raw_start/end、raw_text、clean_start/end、alignment；删除块保留原因 |
| CleanDocument | clean_counts | 输入／输出字符、块、删除等统计，展开查看具体计数 |

所有输入字段继续随行保留。清洗去掉能规则识别的外壳，不能仅因包含链接就删除知识，也不负责判定概念是否相关、来源是否可靠。`map_cached` 保存按输入与版本定位的结果；`checkpoint` 是执行和落盘边界。

新增 `knowledge_eligibility`：status 为 eligible_for_identity 或 pending；reasons 是待处理原因；next_actions 指明补全文、重新获取、清除界面、解析源标记、核对语言等任务。eligible_for_identity 只表示可以送身份模型，不表示已核验概念归属或事实。pending 原文保留，但身份与材料选择算子均拦截。

本轮清洗使用 mwparserfromhell 0.7.2 解析 Wiki 语法树。clean_blocks 增加：references（reference_id/name/raw_text/content，原始引文标识、名称、标记与注释正文）、templates（name/parameters/raw_text，模板名称、参数和原文）、unsupported_markup（尚不能解释的语义标记）。decision=defer 的块不进入 clean_text；原文仍保留，counts.deferred 统计暂缓块。partial_markup_deferred 表示部分可读正文可继续，不能代表整篇已处理完成。

原生 Wiki 的 sections 边界也用于识别标题，避免只认 Markdown 标记；原始正文串接方式不变。部分解析且剩余正文不足的页面会标 partial_body_context_requires_review，要求核对被暂缓块中的必要上下文。

</details>

<details>
<summary>4．检查图片字节：图片独立扩列</summary>

输入：images_selected，重点使用 path、sha256。

输出保留全部输入字段，增加 byte_status（verified_bytes、not_local、hash_mismatch 等）和 byte_details（实际路径、解码／哈希检查结果、错误详情）。这一步核验文件可用性，不产生像素内容判断，也不证明图片支持哪条知识。

</details>

<details>
<summary>5．按概念统计资料覆盖：概念扩列</summary>

输入：入选概念、已完成的文档／图片 Dataset。通过 `.join()` 关联键并 `.reduce_by_key()` 计数，必须等两个分支都已落盘。

输出 concepts_ready 增加 document_count、image_count（关联资料数）、readable_documents、verified_images（读取成功／字节核验成功数）、material_status（有资料或当前读取范围内没资料）、knowledge_status（not_extracted）。这里的身份状态仍是 source_only。零资料不表示概念本身不存在，也不表示全网无资料。

</details>

<details>
<summary>6．按概念分批汇集文档／图片：首次嵌套</summary>

输入：concepts_ready、documents_processed、images_processed。两条资料链独立处理完，再用 demiflow `.group_batches("concept_ref", max_rows=32)`，并左关联概念。

输出 knowledge_inputs 保留概念列，增加 materials、group_index、group_last。materials 每项含 concept_ref、doc_id 或 image_id、material_type（documents／images）、material（完整扩列文档／图片）。group_index 从 0 开始，group_last 表示该概念最后一批。无资料概念保留，可能没有 materials 字段。

这是执行分批，不是语义拆分；**跨批身份和知识联合整合尚未实现**，各批不能假装代表完整概念。大文档也仍需进一步实现章节提取与联合整合。

</details>

<details>
<summary>核对概念身份、判定资料归属</summary>

输入 cleaned_materials 和来源身份字段；输出 identity（status、target_label、reason、accepted_material_ids、rejected_materials、identity_groups、call）、identity_materials、identity_unexamined。blocked 记录阻塞原因。当前有限文字预览与图片元数据不能视作完整身份／像素审核。

所有阶段保留输入及来源；失败单独记录，不覆盖历史成功调用。

新版身份输入给出首／中／尾片段及准确 start/end，图片明确 pixels_provided=false。输出 material_reviews 逐材料包含 relation、basis、quote、reason；same_identity 与 related_context 可以进入 accepted；后者仅表示有目标相关知识，不合并概念身份。无关、身份不明、不可读不得接受。metadata 不能冒充看图，quote 必须出自实际提供的片段、标题或图注。尚未执行的片段和其他批次不算已核验。

身份原始引文不匹配时，在protocol_issues保存模型原判，仅该材料转待核验；不会伪造引文，也不会阻塞其他已通过的材料。
identity_images 独立控制图片元数据预览数量（默认12），不再与实际图片支持 max_images（默认2）混用；优先预览字节已在本地的图片，所有未预览材料仍在 identity_unexamined。

先给不同规范化页面预览机会，再用剩余预算查看同页其他版本；未预览的版本不会因为同页代表被接受而自动获得身份确认。

身份输出的 `protocol_issues` 保存协议问题和原判断。只有逐材料判断完整覆盖全部输入、但接受／拒绝汇总列表漏项时，才根据已有明确 relation 补齐汇总；不推断未查看材料身份。

</details>

<details>
<summary>去除重复资料、按章节选择完整正文块</summary>

输入：身份接受的文档及 clean_blocks（含章节、原文和清洗后定位），以及图片字节结果。输出仍为 material_pack，不改变 Dataset 编排。

- passages：按章节轮流选择能容纳的完整块，保留 source_id、text、start/end、original_chars、source_family、provenance、document_sha256、quote_basis、cleaning_version、raw_source_sha256、source_locator、source_blocks。start/end 是清洗全文中的位置，source_blocks 提供对应原文；片段可能不连续。
- sections：该片段所属的来源章节标题；selection：whole_blocks_across_sections，表示本批选取策略，不是全文覆盖。
- reference_notes：本片段对应的原始引文或注释；包含 reference_id、name、raw_text、content 或 parameters。模型需核对其中条件，注释字符也计入输入预算。
- omissions：未选文档、超预算的完整块、未解析语义块；保存 material_id、reason、start/end 或 raw_start/raw_end、block_id 和必要的 next_action，随最终候选导出。
- duplicates：重复原文／图片的来源关联；images：本批字节可用图片；image_gaps：尝试后仍不可用的图片；coverage：明确本批覆盖范围。

仍有每批文档和字符预算；未完成全文逐章节提取及跨批整合。程序不能用来源排序代替可靠性审核。

source_family 为规范化页面地址：同页的移动版、语言变体路径和版本URL归为同页；不同页面也不自动表示独立来源。document_selection记录每页的material_id、canonical_page、direct_subject（主体与目标一致）、traceable_citations（可定位引文标识数量）、citation_band（排序档位，上限3）、encyclopedia_page（百科页面类型线索）与selected。排序先看目标关系和引文线索，再看百科页面类型，最后用页面地址保证可复现；这些不是来源可靠性认证，未选页面保留后续探索。

超出图片支持预算的已关联图片也进入omissions，reason=image_support_budget、next_action=check_remaining_image_support；与image_gaps中的获取失败分别记录。

</details>

<details>
<summary>联合提取知识陈述</summary>

输入本次 passages；输出 extraction.facts、unresolved_conflicts、coverage_note 及调用记录。fact_id 标识候选，statement 是陈述，conditions/exceptions 是条件与例外，evidence 的 source_id/quote 是可定位引文。

所有阶段保留输入及来源；失败单独记录，不覆盖历史成功调用。
新增deferred_facts：每项保存原始fact、reasons和next_action；不匹配引文、未经核验的数值换算、词源对象丢失会单独暂缓，不牵连其他有效引文候选。reasons中引文／数值检查用文字原因，冲突检查另保存conflict_id/basis对应依据。

coverage_note由程序声明本批未穷尽，model_coverage_note保留模型原始范围意见但不作为排除规则；remaining_knowledge_review为每个输入片段保存source_id、material_id、start/end与继续审查动作。地域、民族、词源、方法等知识不会因为模型称为“非核心”而被关闭处理。

数值检查允许经完整年月日匹配的英文月份翻译；不允许据此放行其他新增数字。引文未含对应条件连接词时，额外“否则”后果会暂缓。这只是已覆盖错误的检查，不是通用语义证明。

</details>

<details>
<summary>比较重复、互补、条件差异与矛盾</summary>

输入同次提取多份来源及 extraction；输出 knowledge，保留 facts、unresolved_conflicts、changes、coverage_note。机器复核不能自动升级为核验知识；未解争议不应混进确定输出，现有模型仍可能违反这一点，需要审核。

所有阶段保留输入及来源；失败单独记录，不覆盖历史成功调用。

新版知识复核将争议事实移到 knowledge.deferred_facts，保存 fact、reasons、next_action；facts 只保留未被当前争议涉及的候选。未报告 affected_fact_ids 或改名导致对应不明时，保守暂缓共享争议来源的陈述并明确记录原因。提取阶段已经报告的冲突不能通过后一步漏报而消失；语义消解仍需证据审核，不用多数票消解。
新增deferred_facts：每项保存原始fact、reasons和next_action；不匹配引文、未经核验的数值换算、词源对象丢失会单独暂缓，不牵连其他有效引文候选。reasons中引文／数值检查用文字原因，冲突检查另保存conflict_id/basis对应依据。

coverage_note由程序声明本批未穷尽，model_coverage_note保留模型原始范围意见但不作为排除规则；remaining_knowledge_review为每个输入片段保存source_id、material_id、start/end与继续审查动作。地域、民族、词源、方法等知识不会因为模型称为“非核心”而被关闭处理。

数值检查允许经完整年月日匹配的英文月份翻译；不允许据此放行其他新增数字。引文未含对应条件连接词时，额外“否则”后果会暂缓。这只是已覆盖错误的检查，不是通用语义证明。

</details>

<details>
<summary>核验图片对具体知识的支持</summary>

输入知识候选和可用图片；输出 image_evidence：图片内容描述、逐图片×知识的支持结果及调用信息。必须区分原图注、补充描述、支持区域／范围／局限；本地真实图像分支已调用并保存逐图×知识矩阵；COS远程获取仍未验收。机器支持判断仍需核对，尤其是乐器类型、材质、绘画与实物证据的区别。

所有阶段保留输入及来源；失败单独记录，不覆盖历史成功调用。

</details>

<details>
<summary>保存知识候选与审核状态</summary>

输入前面累计结果；输出 export：case_id、concept_id、request、status、blocked、identity、facts、unresolved_conflicts、image_evidence、scope。同时保存 candidates JSON。状态仍是机器候选／阻塞，不是正式干净知识库验收，也不是 V4 试题。

所有阶段保留输入及来源；失败单独记录，不覆盖历史成功调用。

导出增加 deferred_facts、material_followups 和 image_followups；无可继续候选时 status 为 pending_evidence_or_review。图片支持缺失只产生补图／核验任务，不删除有文字证据的知识。后续任务目前保存为可追溯待办，尚不自动执行新的采集。
数值检查允许经完整年月日匹配的英文月份翻译；不允许据此放行其他新增数字。引文未含对应条件连接词时，额外“否则”后果会暂缓。这只是已覆盖错误的检查，不是通用语义证明。

</details>



### 原生调用与缓存补充
| 内容 | 输入 | 输出/保存位置 |
|---|---|---|
| demiflow文件reader | JSONL/gzip、JSON数组或文本流 | value原始值、path文件路径、row从1开始的行号/项号、error解析错误、raw出错原文；随后由ReadSource解释采集字段 |
| Prepare算子 | 概念材料批次/上阶段输出 | identity_prompt、extract_prompt、consolidate_prompt或evidence_prompt；图片另外为evidence_images、prepared_image_roles |
| map_prompt_async | payload绑定上述prompt列，图片阶段另绑定images | prompt_result模型对象；prompt_call含request_path/response_path文件、usage原始用量、reused是否回放、attempts尝试记录；prompt_error含type/detail/call |
| Apply算子 | 原资料、prompt_result、prompt_call或prompt_error | identity/extraction/knowledge/image_evidence；不合格保留blocked和原因；成功移除临时列 |
| map_cached | 业务actor、当前行、version、cache_dir | 同一完整输入/版本的逐行缓存；回放元数据改变时可生成新缓存，不覆盖旧行 |
| checkpoint | 一个阶段Dataset与冻结version | datasets/knowledge_阶段.jsonl；中断的partial保留，完整文件可直接read_json |

原生read_records输出：value为解码内容，path为文件路径，row为从1开始的行号/JSON项号，error为解析错误或null，raw为出错原文，snapshot为读取时文件快照。read_*保存全部扫描行，input_*保存转换后的业务行。源报告中的invalid_rows只统计解码失败；非对象JSON行可在read_*中查看。


最终sink为knowledge_base.jsonl，每行一个概念材料批：concept保存概念及身份来源，documents保存原始文档/清洗内容/定位，images保存图片元数据、字节路径及选中图片ID，knowledge保存候选/暂缓/争议/支持关系/补证任务，audit保存处理范围与调用追踪。图片字节仍通过原路径引用，不重复拷贝。多材料批尚未自动合并成全概念知识。

images保留全部原始关联图；image_selection.original_material_ids列出原始图ID，reviewed_image_ids列出已完成支持检查的图，supporting_images只保存支持保留知识的full/partial关系和material_id引用。原始关联、已检查、有效支持候选三者不等同；有效支持仍非人工审核通过。

忠实性审核输入：`knowledge.facts`、`knowledge.deferred_facts[].fact`（全部候选，含 statement/conditions/exceptions/evidence），`material_pack.passages`（完整入选片段及定位）。输出 `fidelity_reviews`：每次审核的 `reviewer` 审核者、`input_hash` 输入内容哈希、`passage_ids` 所看片段、`call` 完整请求响应路径、`scope` 未核验边界、`reviews` 逐条结论；每项 `fact_id` 知识ID、`verdict` faithful/unsupported/uncertain、`reason` 依据、`issues` 问题列表。问题字段：`claim` 陈述中出错部分（遗漏可空）、`source_id` 片段ID、`source_quote` 连续原文、`kind` 问题类型、`reason` 差异说明。通过只表示来源忠实，不代表来源为真；修订后必须重新核验。

新增 `reviews[].source_conditions`：来源必要限定的 `source_id`/`quote`、是否保留的 `preserved` 和判断依据 `reason`。提示输入里的 `review_facts[].evidence` 仅含 `source_id` 与程序检查的 `quoted_text_matches`，不再把候选错误引文传作原文；原始引文仍完整保存在知识条目中。`original_facts_sha256` 将完整原候选绑定到本次审核；真实原文来自 `passages.text`。


<details><summary>原文优先分支：每步输入、输出及字段</summary>

| 算子 | 输入 | 输出与字段含义 |
|---|---|---|
| BuildSourceBlocks | identity_materials、accepted_material_ids、cleaning.blocks/text | source_units：全部接受文档的保留块；block_scope：文档覆盖、清洗排除/暂缓、身份未查看范围 |
| BatchSourceBlocks | source_units、概念身份 | 每行一个block_prompt；batch_id标识请求，units为完整文本及上下文 |
| ApplyBlockSelection | block_prompt、模型结果/错误/调用记录 | block_decisions：unit_id、decision（selected/deferred/excluded）、reason、relation（direct/background/unrelated/uncertain）、relation（direct/background/unrelated/uncertain）、source_check_needed（需查来源，不代表真假）、protocol_valid；block_calls保存完整调用路径及原回答 |
| BuildVerbatimCandidates | 原文块join全部批次决定 | knowledge.facts/deferred_facts；statement与evidence.quote由程序逐字复制；statement_mode=verbatim_source_block；conditions/exceptions为空仅表示未另行拆列，条件仍在原文；truth_status=not_verified |
| BatchSourceComparisons | 全部保留与暂缓候选、原文 | compare_prompt：分组内和组间两两比较请求，fact_id对应候选；无按主题提前排除的组合 |
| ApplySourceComparison | 比较请求、模型结果/错误 | source_comparisons：看过的fact_ids、protocol_valid、result、call、error；result.pairs记录duplicate/complement/condition_difference/potential_conflict及依据；issues记录具体候选问题 |
| ApplyComparedCandidates | 候选join全部比较响应 | 不改写原文；分歧、缺上下文、比较失败涉及项暂缓；source_relations保留关系，block_comparison_coverage记录候选数、应覆盖和已通过协议的候选对数 |

source_units每项：unit_id/source_id为本原文片段ID；material_id关联原文档；text为清洗全文的连续片段；start/end是清洗全文位置；sections为章节路径；context_before/after为同章节相邻完整块；reference_notes为来源原注释；source_family为规范化页面地址，不能据此判独立来源；source_blocks保留raw_start/end和原始文本；document_sha256/raw_source_sha256绑定清洗文本及原文版本；provenance/source_locator保存来源定位；cleaning_version记录清洗版本。

source_relations不自动合并或删除来源；potential_conflict是模型提出的待核查差异，不是已经证实的矛盾。
原文优先候选没有生成式改写，但仍可能选择失误、保留宣传主张或漏掉上下文。
最终audit.source_block_review保存分块覆盖、逐项选择、全部模型调用、来源比较与协议覆盖；原始文档图片仍在同一五列表格中。
</details>

严格正文版本（body_only=True）：先将推荐区、引用/外链目录、广告区、独立编辑按钮、多卡片混合块移出模型输入；原文及定位保留在block_scope.documents[].structurally_filtered。正文内第三方引用链接不因域名不同而过滤，References仍作为来源线索保留。标题单独存在时只作为sections元数据，不单独打分。仅有“目录”章节标签不足以排除正文，防止采集标题边界错误误伤导言。

严格评分：relevance_score（0–3，整块相关性）、usability_score（0–3，正文可用性，非事实真伪）、standalone（必要条件是否已在正文）、mixed_content（有无无关内容/外壳混入）；两个分数都3且standalone=true、mixed_content=false才准入。该阈值是本次偏高精度试验配置，模型打分仍须抽查；不把较低分内容从原始资料中删除。


## 新筛选算子回放验证

以下是114张图片的已保存真实模型响应，经当前demiflow算子重放的结果。零新增模型调用，验证判断合并与字段处理；不是新端到端知识结果。56张入选仍含物种身份未认证图片，双模型一致不等于事实正确。

In [ ]:
from pathlib import Path
import json
from IPython.display import display
import pandas as pd
check = json.loads(Path('/yzp/zhaozy/yangzepeng/0905/demiwtg/state/curation/v4/dual_image_pipeline_check_v1/validation.json').read_text())
display(pd.DataFrame([{'入选':check['decisions']['keep'],'排除':check['decisions']['exclude'],'暂缓':check['decisions']['pending'],'新模型调用':check['new_model_calls'],'初筛批次':check['primary_batches'],'需复核批次':check['review_batches']}]))
print('温度计：暂缓；空心管：排除。真实旧响应回放通过；尚未实跑新级联，也未重新生成最终知识。')